# Итоговый эксперимент: write → next read
Первый день: обучение и выбор по validation. Дни 2–4: замороженные модели и медианы для usr_1 и prxy_1. День 2 prxy_1 — воспроизведение уже проверенного периода. Здесь только просмотр результатов, без обучения.

Kernel: `.venv/bin/python` после `uv sync --extra notebook`. Корень проекта определяется автоматически при открытии из notebooks/. Главный вывод: преимущество зависит от нагрузки; отрицательный результат usr_1 сохранён.

In [ ]:
from pathlib import Path
import json, sys
from IPython.display import Markdown, display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'read_prediction').is_dir() and (p / 'pyproject.toml').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from read_prediction.paths import REPORTS
RESULTS = ROOT / 'results' / 'final-study'
assert (RESULTS / 'verification.json').exists(), 'Нужны результаты проекта'

In [ ]:
display(Markdown((REPORTS / 'summary.ru.md').read_text()))

## Подробный отчёт
RU: `reports/report.ru.md`. EN: `reports/report.en.md`. Простое объяснение: `reports/simple-summary.ru.md`. Короткий английский итог: `reports/summary.en.md`. Все метрики: `reports/metrics.csv`. Прогнозы: `results/final-study/<workload>/day<2|3|4>.predictions.npz`.

Метрики относятся к записям с чтением до конца своего дня. Неизвестные интервалы не объявлены бесконечными или нулевыми.

In [ ]:
for workload in ('usr_1', 'prxy_1'):
    record = json.loads((RESULTS / workload / 'results.json').read_text())
    print(workload, 'selected:', record['frozen']['selection']['method'])
    for day, result in record['days'].items():
        baseline = result['metrics']['address_median']['mae_seconds']
        selected = result['metrics']['selected']['mae_seconds']
        print(f'  Day {day}: median MAE={baseline:.2f}s, selected MAE={selected:.2f}s; {result["evaluation_role"]}')
print(json.dumps(json.loads((RESULTS / 'verification.json').read_text()), indent=2))

## Воспроизведение
Команды для чистого каталога — в README.md и report.en.md. Проверка готовых артефактов: `uv run python -m read_prediction.checks.study`. Формирование отчётов и проверка notebook: `uv run --extra notebook python -m read_prediction.reporting --check-notebook`. Исходные архивы и прошлые этапы сохраняются; новые fits после просмотра будущих дней не выполнялись.